## Lecture 5: Parallel Computing 2 — Granularity, Load Balancing & Map-Filter-Reduce

### ****Exercise 1:** Overhead vs. Granularity (25 min)**

****Goal:** Observe how chunk size $L$ affects parallel overhead using Monte Carlo $\pi$.**

In [ ]:
# From slide 23


from multiprocessing import Pool
import random
import time
import os


def monte_carlo_chunk(num_samples):
    """Estimate pi contributions for num_samples random points."""
    inside = 0
    for _ in range(num_samples):
        x, y = random.random(), random.random()
        if x*x + y*y <= 1:
            inside += 1
    return inside


def test_granularity(total_work, chunk_size, n_proc):
    n_chunks = total_work // chunk_size
    tasks = [chunk_size] * n_chunks
    t0 = time.perf_counter()
    if n_proc == 1:
        results = [monte_carlo_chunk(s) for s in tasks]
    else:
        with Pool(processes=n_proc) as pool:
            results = pool.map(monte_carlo_chunk, tasks)
    return time.perf_counter() - t0, 4 * sum(results) / total_work


if __name__ == '__main__':
    total_work = 1_000_000
    n_proc = os.cpu_count() // 2
    chunk_sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
    print(f"{'L':>12} | {'serial (s)':>12} | {'parallel (s)':>12}")
    for L in chunk_sizes:
        t_ser, _ = test_granularity(total_work, L, n_proc=1)
        t_par, pi = test_granularity(total_work, L, n_proc=n_proc)
        print(f"{L:12d} | {t_ser:12.4f} | {t_par:12.4f} pi={pi:.4f}")

****Setup:****

- **Total work $N$ = const (same total compute regardless of $L$)**

- **Vary $L$ from $10$ to $1,000,000$ samples per chunk**

- **Test with `P=1` (serial) and `P=os.cpu count()//2` (parallel)**

- **Print elapsed time for each configuration**

****What to look for:****

- **Serial time: stays roughly flat (computation is constant, no IPC overhead)**

- **Parallel time: U-shaped curve — high overhead at small $L$, poor balance at large $L$**

- **Where is the optimal $L$ for your machine?**

****Done?** Identify optimal chunk size → discuss with a neighbour**

### ****Exercise 2:** Serial pipeline → parallel map**

****Data:** $N = 1,000,000$ random integers in $[10, 100]$**

****Pipeline:** map (subtract 7) → filter (keep odd) → reduce (sum)**

In [ ]:
# From slide 25

# Setup — data and worker function (module-level for pickling):
import random, time
from functools import reduce
from multiprocessing import Pool

N = 1_000_000
data = [random.randint(10, 100) for _ in range(N)]

def subtract_seven(x):
    return x - 7

#### ****Part 1:** — Serial (10 min):**

In [ ]:
# From slide 25

# Part 1 — serial pipeline (map / filter / reduce chained):
t0 = time.perf_counter()
result_ser = reduce(lambda a, b: a + b,
                    filter(lambda x: x % 2 == 1,
                           map(subtract_seven, data)))

t_serial = time.perf_counter() - t0

- **Implement the pipeline using `map()`, `filter()`, `reduce()` built-ins**

- **Run and verify the result**

#### ****Part 2:** — Parallel (15 min):**

In [ ]:
# From slide 25

# Part 2 — replace `map()` with `Pool.map()`:
t0 = time.perf_counter()
with Pool() as pool:
    mapped = pool.map(subtract_seven, data)
result_par = reduce(lambda a, b: a + b,
                    filter(lambda x: x % 2 == 1, mapped))
t_parallel = time.perf_counter() - t0

print(f"Serial:     {t_serial:.4f}s result={result_ser}")
print(f"Parallel:   {t_parallel:.4f}s result={result_par}")
print(f"Speedup:    {t_serial / t_parallel:.2f}x")

- ****Before running — predict:** will `Pool.map()` be faster? Why or why not?**

- **Replace `map(subtract_seven, data)` with `pool.map(subtract_seven, data)`**

- **Time both versions and compare**

**After running:**

- **Was your prediction correct?**

- **How does this compare to the overhead curve you saw in E1?**

****Done?** Discuss with a neighbour → move on to MP2 milestones**

### ****Milestone 1:** Chunked Mandelbrot (20 min)**

****Goal:** Decouple the number of chunks from the number of workers.**

In [ ]:
# From slide 27

# From L04 (unchanged — add `cache=True`` if not already there):
@njit(cache=True)
def mandelbrot_pixel(c_real, c_imag, max_iter): ...
@njit(cache=True)
def mandelbrot_chunk(row_start, row_end, N, x_min, x_max, y_min, y_max, max_iter): ...
def _worker(args): return mandelbrot_chunk(*args) # plain Python, must be module-level

In [ ]:
# From slide 27

# Extend your L04 mandelbrot parallel to (include n chunks):
def mandelbrot_parallel(N, x_min, x_max, y_min, y_max,
                        max_iter=100, n_workers=4, n_chunks=None):
    if n_chunks is None:
        n_chunks = n_workers
    chunk_size = max(1, N // n_chunks)
    chunks, row = [], 0
    while row < N:
        row_end = min(row + chunk_size, N)
        chunks.append((row, row_end, N, x_min, x_max, y_min, y_max, max_iter))
        row = row_end
    tiny = [(0, 8, 8, x_min, x_max, y_min, y_max, max_iter)]
    with Pool(processes=n_workers) as pool:
        pool.map(_worker, tiny)         # warm-up: load JIT cache in workers
        parts = pool.map(_worker, chunks)
    return np.vstack(parts)

****What to do:****

- **Add an `n_chunks` parameter to `mandelbrot_parallel` (default: `n_chunks = n workers`)**

- **Verify output still matches the serial result from L04**

- **Re-run the L04 worker-count benchmark with `n_chunks = 4 × n workers`; note any change**

****Done?** Verify result matches serial output → commit**

### ****Milestone 2:** Optimal Chunk Size (15 min)**

****Fix `n_workers` at your L04 optimum. Sweep `n_chunks`, measure wall time, and compute LIF = $ p \cdot T_p/T_1 − 1 $.****

****Suggested configurations (`n_workers` = your best):****

| **n_chunks** | **time (s)** | **vs. 1×** | **LIF** |
| -------------- | ------------ | ---------- | ------- |
| 1× n workers   | —            | baseline   | —       |
| 2× n workers   | —            | —          | —       |
| 4× n workers   | —            | —          | —       |
| 8× n workers   | —            | —          | —       |
| 16× n workers  | —            | —          | —       |

****Tips:****

- **Use `statistics.median()` of 3+ runs**

- **Include warm-up before timing**

- **LIF minimum = sweet spot**

- **Typical sweet spot: 4–8× `n_workers`**

****Done?** Record optimal `n_chunks` and LIF in performance notebook (MP2) → commit**

### ****Milestone 3:** Comprehensive Analysis (20 min)**

****Create a full performance comparison at $1024×1024$, `max_iter=100`:****

#### ****1 Speedup table:****

| **Implementation** | **Time (s)** | **Speedup** |
| ------------------ | ------------ | ----------- |
| Naive Python       | —            | 1x          |
| Numpy              | —            | —           |
| Numba (@njit)      | —            | —           |
| Parallel (opt.)    | —            | —           |

#### ****2. Speedup vs. core count:****

- **Actual vs. ideal (linear)**
- **Back-solve implied $s = \frac{1/S_p^∗ − 1/p^*}{1 − 1/p^*}$**
- **Is your s smaller than your L04 result?** - Better load balance ⇒ lower apparent $s$?

#### ****3. Brief recommendation:****

- **What settings give the best time?**

- **Is parallelisation worth it on your hardware?** - Compare speedup and LIF to your serial
Numba baseline

- **Write 2–3 sentences in your performance notebook**

#### ****4. Submit to Performance Tracker:****

- **Enter your best multiprocessing time on Moodle - (1024×1024, Implementation: “Multiprocessing”)**

****Done?** Record all results in performance notebook (MP2) → commit**